# Silver: parse the PDFs with `ai_parse_document`

Task `silver_parse` of the job `ingest_financebench`. Runs the AI parser on every part in `bronze_pdf_parts` and
saves the result in `silver_parsed`, **one row per part**: `path`, `part_no`, `page_offset`, `parsed` and `parsed_at`.

`parsed` is the parser's output (a VARIANT): the `elements` of the part in reading order, each with a `type`
(title, section_header, text, table as HTML, page_header, ...), its `content` and its page in `bbox[0].page_id`. The
real page of an element in the original PDF is `page_offset + page_id + 1`.

**Only complete parses are saved.** The parser sometimes answers pages with an error (`Request rate limit exceeded`,
`Internal error`), and a part with missing pages would silently lose text. So a part is saved only if it has
elements and no `error_status`. A part that is not saved is still **missing**, and **the next run parses it again**:
there is no retry job and no log table. A rerun parses only what is missing, so a saved part is never paid for twice.

**A few PDFs at a time** (`pdfs_per_batch`). Parsing many PDFs at once made the parser rate-limit, which is also why
bronze cuts long PDFs into parts of 50 pages: a failed part costs at most 50 pages to parse again.

**Parts that never succeed** (the two Intel 8-Ks fail with `Document conversion failed`) are tried again on every
run, because nothing records that they failed. See what is missing with the query in the last cell.

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")
dbutils.widgets.text("pdfs_per_batch", "2")  # PDFs parsed at once, small so the parser's rate limit is not hit

catalog = dbutils.widgets.get("catalog")
parts_table = f"{catalog}.{dbutils.widgets.get('bronze_schema')}.bronze_pdf_parts"
parsed_table = f"{catalog}.{dbutils.widgets.get('silver_schema')}.silver_parsed"
pdfs_per_batch = int(dbutils.widgets.get("pdfs_per_batch"))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{dbutils.widgets.get('silver_schema')}")

In [ ]:
# The parts that are not saved yet.
missing = spark.table(parts_table)
if spark.catalog.tableExists(parsed_table):  # the first run creates the table (saveAsTable below)
    missing = missing.join(spark.table(parsed_table).select("path", "part_no"), ["path", "part_no"], "left_anti")

paths = [r.path for r in missing.select("path").distinct().orderBy("path").collect()]
print(f"{len(paths)} PDFs have parts to parse.")

# Parse a few PDFs at a time and save the complete parses of each batch. The parse is billed and runs once: the filter
# and the saved column both read the same parser call.
for i in range(0, len(paths), pdfs_per_batch):
    batch = paths[i:i + pdfs_per_batch]
    (
        missing.filter(F.col("path").isin(batch))
        .select("path", "part_no", "page_offset", F.expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed"))
        .filter(
            "coalesce(size(cast(parsed:document:elements AS array<variant>)), 0) > 0"  # it has elements
            " AND coalesce(size(cast(parsed:error_status AS array<variant>)), 0) = 0"  # and no page failed
        )
        .withColumn("parsed_at", F.current_timestamp())
        .write.mode("append").saveAsTable(parsed_table)
    )
    print(f"PDFs {i + 1}-{i + len(batch)} of {len(paths)} done")

In [ ]:
# What is saved, and what is still missing (the next run parses it again).
display(spark.sql(f"""
SELECT (SELECT count(*) FROM {parts_table}) AS parts, (SELECT count(*) FROM {parsed_table}) AS parts_parsed
"""))
display(spark.sql(f"""
SELECT p.path, p.part_no, p.pages
FROM {parts_table} p LEFT ANTI JOIN {parsed_table} s ON s.path = p.path AND s.part_no = p.part_no
ORDER BY p.path, p.part_no
"""))